In [2]:
# 1. Imports
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from tensorflow.keras.layers import TextVectorization, Embedding, LSTM, Dense, Input
from tensorflow.keras.models import Model

# 2. Load the dataset
dataset = load_dataset("nickmuchi/financial-text-combo-classification", split="train")
df = pd.DataFrame(dataset)

# 3. Prepare inputs
texts = df["text"].astype(str).values
labels = df["label"].astype(int).values
num_classes = len(set(labels))

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# 5. Text Vectorization
max_vocab = 20000
max_len = 100
vectorizer = TextVectorization(
    max_tokens=max_vocab,
    output_mode='int',
    output_sequence_length=max_len
)
vectorizer.adapt(X_train)

# 6. Create tf.data.Dataset
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

def to_dataset(x, y):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.shuffle(1000).batch(batch_size).prefetch(AUTOTUNE)
    return ds.map(lambda x, y: (vectorizer(x), y))

train_ds = to_dataset(X_train, y_train)
test_ds = to_dataset(X_test, y_test)

# 7. Build LSTM model
inputs = Input(shape=(max_len,), dtype=tf.int32)
x = Embedding(input_dim=max_vocab, output_dim=128, mask_zero=True)(inputs)
x = LSTM(64)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 8. Train the model
model.fit(train_ds, validation_data=test_ds, epochs=5)

# 9. Evaluate
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc:.4f}")

# 10. Predict new samples
def predict(texts):
    sequences = vectorizer(texts)
    probs = model.predict(sequences)
    return np.argmax(probs, axis=1)

# Example usage
samples = [
    "Credit card declined due to overlimit",
    "Loan approved for customer",
    "Salary credited to checking account"
]
predictions = predict(samples)
for txt, label in zip(samples, predictions):
    print(f"'{txt}' → Label: {label}")

Epoch 1/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - accuracy: 0.6705 - loss: 0.8011 - val_accuracy: 0.8164 - val_loss: 0.5034
Epoch 2/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.8866 - loss: 0.3180 - val_accuracy: 0.8289 - val_loss: 0.5289
Epoch 3/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.9611 - loss: 0.1226 - val_accuracy: 0.8120 - val_loss: 0.6149
Epoch 4/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.9822 - loss: 0.0631 - val_accuracy: 0.8331 - val_loss: 0.6691
Epoch 5/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - accuracy: 0.9911 - loss: 0.0325 - val_accuracy: 0.8284 - val_loss: 0.7576
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8293 - loss: 0.7591
Test Accuracy: 0.8284
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
'Credit card declined due to overlimit' → Label: 0
'Loan approved for customer' → Label: 1
'Salary credited to checking account' → Label: 1
